# Dagu: train the cell classifier on Google Colab

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Free Colab does not guarantee a GPU or a session length (typically up to about 12 hours, and it disconnects after idle time), so this notebook saves a checkpoint to Google Drive after every epoch. If the session dies, rerun cells 1 to 6, then run the training cell again: it resumes.

**Before you train:** your mentor must have approved the dataset (cell 5). Research prototype, not a diagnostic device.

**One-time setup on Drive** (`My Drive/dagu/`): upload `dagu-malaria.zip` (this repo). Optionally upload `cell_images.zip` too, so you do not download it every time.

In [ ]:
# 1. Which hardware did we get?
!nvidia-smi
import os, torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available(), "| cpus:", os.cpu_count())

In [ ]:
# 2. Mount Drive so checkpoints and results survive a disconnect.
#    If Drive cannot be mounted (for example in the VS Code Colab extension), we fall back to TEMPORARY storage:
#    then download your results before the session ends (last cell).
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/dagu'
except Exception as e:
    print('Drive not available, using temporary storage:', e)
    DRIVE = '/content/dagu_tmp'
RUN = f'{DRIVE}/runs/run1'
os.makedirs(RUN, exist_ok=True)

In [ ]:
# 3. Get the code onto the Colab runtime's local disk, then install the few extra packages.
#    Option A: your GitHub repo (best for a VS Code workflow: edit, push, rerun this cell).
#    Option B: leave REPO_URL empty and use dagu-malaria.zip from Drive.
REPO_URL = ''   # e.g. 'https://github.com/<you>/dagu-malaria.git'  (never put a token in a notebook you share)
if REPO_URL:
    !git clone -q {REPO_URL} /content/dagu-malaria || (cd /content/dagu-malaria && git pull -q)
else:
    !unzip -q -o {DRIVE}/dagu-malaria.zip -d /content
%cd /content/dagu-malaria
!pip install -q onnx onnxruntime mlflow pyyaml
os.environ['PYTHONPATH'] = '/content/dagu-malaria/src'

In [ ]:
# 4. Run the torch-free tests (about 5 seconds).
!pip install -q pytest httpx python-multipart fastapi
!python -m pytest -q

In [ ]:
# 5. Data. Copy to the LOCAL disk: reading thousands of small files from mounted Drive is very slow.
APPROVED = False   # set True ONLY after your mentor approved this dataset and you checked its terms
ZIP_ON_DRIVE = f'{DRIVE}/cell_images.zip'
if not os.path.exists('/content/cell_images.zip'):
    if os.path.exists(ZIP_ON_DRIVE):
        !cp {ZIP_ON_DRIVE} /content/cell_images.zip
    else:
        assert APPROVED, "Set APPROVED = True after mentor sign-off, or upload cell_images.zip to Drive."
        !wget -q -O /content/cell_images.zip https://data.lhncbc.nlm.nih.gov/public/Malaria/cell_images.zip
        !cp /content/cell_images.zip {ZIP_ON_DRIVE}
!unzip -q -o /content/cell_images.zip -d /content/data
# the zip may nest the folders differently, so find the folder holding both class folders
ROOT = next(r for r, d, f in os.walk('/content/data') if {'Parasitized', 'Uninfected'} <= set(d))
print("dataset root:", ROOT)

In [ ]:
# 6. Write a Colab config: data and cache on local disk, checkpoints and reports on Drive.
import yaml
cfg = yaml.safe_load(open('configs/config.yaml'))
cfg['data'].update(root=ROOT, manifest='/content/work/manifest.csv', cache_dir='/content/work/cache')
cfg['train'].update(out_dir=RUN, num_workers=min(4, os.cpu_count() or 2))
cfg['eval'].update(report_dir=f'{RUN}/reports')
os.makedirs('/content/work', exist_ok=True)
yaml.safe_dump(cfg, open('/content/colab_config.yaml', 'w'))
dbg = yaml.safe_load(open('/content/colab_config.yaml'))
dbg['train'].update(out_dir='/content/debug_run', epochs_head=1, epochs_finetune=1, debug_fraction=0.05)
dbg['eval'].update(report_dir='/content/debug_run/reports', bootstrap=50)
yaml.safe_dump(dbg, open('/content/debug_config.yaml', 'w'))
CFG = '/content/colab_config.yaml'; DBG = '/content/debug_config.yaml'

In [ ]:
# 7. Integrity checks, patient-level split, and the image cache (decode and resize ONCE).
!python scripts/verify_dataset.py --config {CFG}
!python scripts/prepare_data.py --config {CFG}
!python scripts/cache_dataset.py --config {CFG}
!cp /content/work/manifest.csv {RUN}/manifest.csv

In [ ]:
# 8. Optional: the simple baseline the CNN must beat (a few minutes).
from dagu import baseline
baseline.run('/content/work/manifest.csv')

In [ ]:
# 9. Quick pipeline check on 5% of the training data (separate output folder, will not touch your real run).
!python -m dagu.train --config {DBG}
!python -m dagu.evaluate --config {DBG} --ckpt /content/debug_run/best.pt

In [ ]:
# 10. Real training. Safe to rerun after a disconnect: --resume continues from the last saved epoch.
!python -m dagu.train --config {CFG} --resume

In [ ]:
# 11. Honest evaluation (threshold from validation, metrics on test) and export.
!python -m dagu.evaluate --config {CFG}
!python -m dagu.export --config {CFG}
print(open(f'{RUN}/reports/metrics.json').read())

## If something goes wrong
| Symptom | Likely cause | Fix |
|---|---|---|
| `cuda available: False` | CPU runtime, or the free GPU pool is empty | Runtime > Change runtime type > T4; try later, or use Kaggle Notebooks |
| Session disconnected mid-training | Idle timeout or session cap | Rerun cells 1 to 7, then cell 10 (resumes) |
| `Image cache does not match the current manifest` | Manifest changed after caching | Rerun `cache_dataset.py` |
| Out of memory on the GPU | Batch too large | Lower `train.batch_size` in the config cell |
| Very slow epochs | Reading from Drive, or no cache | Confirm the data path starts with `/content/`, and the log says `image cache: in use` |

In [ ]:
# 12. Package the small results for download (best.pt, threshold, metrics, manifest, reports).
!cd {RUN} && zip -qr /content/dagu_results.zip best.pt meta.json threshold.json manifest.csv reports
try:
    from google.colab import files
    files.download('/content/dagu_results.zip')
except Exception as e:
    print('Automatic download unavailable here:', e, '\nCopy /content/dagu_results.zip or the run folder from Drive instead.')